# Estimation Consommation Électrique - France

---

## Description du projet

Ce notebook estime la **consommation électrique annuelle** d'un foyer français
et sa **facture EDF estimée**, en le positionnant par rapport aux moyennes
départementales officielles ENEDIS.

---

## Source des données

**Agence ORE / ENEDIS - data.gouv.fr**

- Consommation annuelle résidentielle par département (Électricité uniquement)
- Données officielles ENEDIS, couvre ~95% du territoire français
- Période : **2015 -> 2024** (10 ans)
- **94 départements** métropolitains
- Gratuit, officiel, mis à jour annuellement

URL : `https://opendata.agenceore.fr/api/explore/v2.1/catalog/datasets/consommation-annuelle-d-electricite-et-gaz-par-departement/exports/csv`

---

## Limites documentées

### Ce que le modèle fait bien
- **Situe un foyer** par rapport à la moyenne de son département
- Prend en compte : surface, type de logement, ancienneté, chauffage électrique
- Calibré sur les références **ADEME 2024**

### Ce que le modèle ne fait pas
- Données **agrégées par département** - pas individuelles
- R2 = 0.42 - précision limitée (+/-307 kWh/an, ~7%)
- Pas de DPE, équipements spécifiques, ni habitudes de consommation

### Pourquoi R2 = 0.42 est acceptable
```
On prédit la conso MOYENNE d'un département entier
La conso dépend surtout du taux de chauffage élec
Ce taux est stable dans le temps -> peu de variance exploitable
Le modèle sert à SE SITUER, pas à prédire une facture exacte
```

### Références ADEME 2024
| Type | Sans chauffage élec | Avec chauffage élec |
|:-----|:-------------------:|:-------------------:|
| Studio 45m2 | 2 000-3 000 kWh | 5 000-7 000 kWh |
| Appartement 65m2 | 3 000-4 500 kWh | 7 000-10 000 kWh |
| Maison 90m2 | 5 000-8 000 kWh | 10 000-13 000 kWh |
| Maison 120m2 ancienne | 6 000-10 000 kWh | 14 000-18 000 kWh |

---

## Structure du notebook (11 cellules)

| # | Étape |
|:-:|:------|
| 1 | Imports |
| 2 | Chargement Agence ORE |
| 3 | Filtrage résidentiel électrique |
| 4 | Renommage + nettoyage |
| 5 | Filtrage ENEDIS + imputation |
| 6 | Feature Engineering |
| 7 | EDA |
| 8 | Split temporel + Entraînement |
| 9 | SHAP + Sauvegarde |
| 10 | Fonctions d'estimation + scoring |
| 11 | Interface Gradio |


---
## Cellule 1 - Imports


In [ ]:
# ============================================================
# PROJET   : Estimation Consommation Électrique - France
# SOURCE   : Agence ORE / ENEDIS - data.gouv.fr
# DONNÉES  : Résidentiel électrique 2015-2024 - 94 depts
# MODÈLE   : Ridge Regression (R2=0.42, MAE=307 kWh/an)
# ============================================================

import pandas as pd
import numpy as np
import requests
import io
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
import shap

warnings.filterwarnings('ignore')

set_seed      = 1204
TARIF_KWH     = 0.2516   # tarif réglementé EDF 2024
ABONNEMENT_AN = 120      # abonnement annuel base (EUR/an)

print('OK - Imports chargés')


---
## Cellule 2 - Chargement des données Agence ORE


In [ ]:
url_ore = (
    'https://opendata.agenceore.fr/api/explore/v2.1/'
    'catalog/datasets/consommation-annuelle-d-electricite-'
    'et-gaz-par-departement/exports/csv'
    '-lang=fr&timezone=Europe%2FParis'
    '&use_labels=true&delimiter=%3B'
)

print('Chargement Agence ORE...', end='')
r      = requests.get(url_ore, timeout=30)
df_raw = pd.read_csv(io.StringIO(r.text), sep=';', low_memory=False)
print(f' OK {df_raw.shape}')

print(f'\n Filières : {df_raw["FILIERE"].value_counts().to_dict()}')
print(f'Secteurs : {df_raw["CODE GRAND SECTEUR"].value_counts().to_dict()}')
print(f'Années   : {sorted(df_raw["Année"].unique())}')


---
## Cellule 3 - Filtrage résidentiel électrique

On garde uniquement :
- Filière **Électricité** (pas gaz)
- Secteur **RÉSIDENTIEL** (pas industrie/tertiaire)
- Années **2015-2024** (méthode cohérente - 2011-2014 = agrégation différente)


In [ ]:
df_dept_elec = df_raw[
    (df_raw['FILIERE']            == 'Electricité') &
    (df_raw['CODE GRAND SECTEUR'] == 'RESIDENTIEL') &
    (df_raw['Année']              >= 2015)
].copy()

print(f'Résidentiel électrique 2015-2024 : {df_dept_elec.shape}')
print(f'\n Conso moyenne par année (MWh/foyer) :')
print(
    df_dept_elec.groupby('Année')['Conso moyenne (MWh)']
    .mean().round(3)
)


---
## Cellule 4 - Renommage et nettoyage


In [ ]:
df_res = df_dept_elec.rename(columns={
    'Année'                                : 'annee',
    'Code Département'                     : 'dept',
    'Nom Département'                      : 'nom_dept',
    'Nom Région'                           : 'region',
    'Conso totale (MWh)'                   : 'conso_totale_mwh',
    'Conso moyenne (MWh)'                  : 'conso_moy_mwh',
    'Nb sites'                             : 'nb_foyers',
    'Part thermosensible (%)'              : 'part_thermosensible',
    'Taux de logements collectifs'         : 'taux_collectif',
    'Taux de résidences principales'       : 'taux_residences_princ',
    'Taux de chauffage électrique'         : 'taux_chauffage_elec',
    'DJU à TR'                             : 'dju',
    "Nombre d'habitants"                   : 'nb_habitants',
    'Superficie des logements <30 m2'      : 'surf_moins_30',
    'Superficie des logements 30 à 40 m2'  : 'surf_30_40',
    'Superficie des logements 40 à 60 m2'  : 'surf_40_60',
    'Superficie des logements 60 à 80 m2'  : 'surf_60_80',
    'Superficie des logements 80 à 100 m2' : 'surf_80_100',
    'Superficie des logements >100 m2'     : 'surf_plus_100',
    'Résidences principales avant 1919'    : 'res_avant_1919',
    'Résidences principales de 1919 à 1945': 'res_1919_1945',
    'Résidences principales de 1946 à 1970': 'res_1946_1970',
    'Résidences principales de 1971 à 1990': 'res_1971_1990',
    'Résidences principales de 1991 à 2005': 'res_1991_2005',
    'residences_principales_de_2006_a_2018': 'res_2006_2018',
    'residences_principales_apres_2019'    : 'res_apres_2019',
}).copy()

# Nettoyage : garde seulement les conso réalistes 1-25 MWh/an
nb_avant = len(df_res)
df_res   = df_res[
    (df_res['conso_moy_mwh'] >= 1) &
    (df_res['conso_moy_mwh'] <= 25)
].copy()

# Conversion en kWh
df_res['conso_moy_kwh'] = (df_res['conso_moy_mwh'] * 1000).round(0)
df_res['dept']          = df_res['dept'].astype(str)

print(f'Outliers supprimés : {nb_avant - len(df_res)}')
print(f'Dataset : {df_res.shape}')
print(f'\nConso moyenne (kWh/an) :')
print(df_res.groupby('annee')['conso_moy_kwh'].mean().round(0).astype(int))


---
## Cellule 5 - Filtrage ENEDIS + Imputation

**Pourquoi filtrer ENEDIS uniquement -**
Les 120+ autres opérateurs (régies locales, EDF-SEI...) ne fournissent
pas les données démographiques (taux_chauffage_elec, DJU, surfaces).
ENEDIS = seul opérateur avec ces données, couvre 94 départements.

**Pourquoi imputer -**
Les données démographiques ne sont disponibles que depuis 2018-2024 selon la feature.
On propage la valeur connue vers les années précédentes (le parc immobilier
évolue peu d'une année à l'autre).


In [ ]:
# Filtrage ENEDIS uniquement
df_enedis = df_res[
    df_res['OPERATEUR'].str.upper().str.contains('ENEDIS', na=False)
].copy()

print(f'Après filtrage ENEDIS : {df_enedis.shape}')
print(f'   Depts : {df_enedis["dept"].nunique()} | Années : {sorted(df_enedis["annee"].unique())}')

# Imputation : propagation entre années par département
# (les caractéristiques du parc immobilier évoluent peu)
print('\n Imputation par département...')
df_ml = df_enedis.copy()

cols_a_imputer = [
    'taux_chauffage_elec', 'taux_collectif', 'taux_residences_princ',
    'surf_moins_30', 'surf_30_40', 'surf_40_60', 'surf_60_80',
    'surf_80_100', 'surf_plus_100', 'res_avant_1919', 'res_1919_1945',
    'res_1946_1970', 'res_1971_1990', 'res_1991_2005',
    'res_2006_2018', 'res_apres_2019', 'nb_habitants',
]
cols_a_imputer = [c for c in cols_a_imputer if c in df_ml.columns]

for col in cols_a_imputer:
    df_ml[col] = df_ml.groupby('dept')[col].transform(
        lambda x: x.ffill().bfill()
    )
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())

# DJU : propagation + fallback région
if 'dju' in df_ml.columns:
    df_ml['dju'] = df_ml.groupby('dept')['dju'].transform(
        lambda x: x.ffill().bfill()
    )
    if 'region' in df_ml.columns:
        df_ml['dju'] = df_ml.groupby('region')['dju'].transform(
            lambda x: x.ffill().bfill().fillna(x.median())
        )
    df_ml['dju'] = df_ml['dju'].fillna(df_ml['dju'].median())

print('Après imputation - NaN résiduels :')
for col in ['taux_chauffage_elec', 'dju', 'taux_collectif']:
    if col in df_ml.columns:
        print(f'  {col:<35} : {df_ml[col].isnull().sum()} NaN')
print(f'\nDataset ML : {df_ml.shape}')


---
## Cellule 6 - Feature Engineering

On crée des features synthétiques à partir des données brutes :
- **Surface moyenne** pondérée depuis les tranches de superficie
- **Ancienneté moyenne** du parc immobilier
- **Catégorie climatique** depuis les DJU
- **Offset temporel** pour capturer la tendance 2015-2024


In [ ]:
# Surface moyenne pondérée
cols_surf = [
    ('surf_moins_30', 15), ('surf_30_40', 35), ('surf_40_60', 50),
    ('surf_60_80', 70), ('surf_80_100', 90), ('surf_plus_100', 130)
]
num = sum(df_ml[c].fillna(0) * v for c, v in cols_surf if c in df_ml.columns)
den = sum(df_ml[c].fillna(0)     for c, v in cols_surf if c in df_ml.columns)
df_ml['surface_moy_dept'] = num / den.replace(0, np.nan)
df_ml['surface_moy_dept'] = df_ml['surface_moy_dept'].fillna(
    df_ml['surface_moy_dept'].median()
)
print('Surface moyenne département')

# Ancienneté moyenne du parc
cols_age = [
    ('res_avant_1919', 1900), ('res_1919_1945', 1932),
    ('res_1946_1970', 1958), ('res_1971_1990', 1980),
    ('res_1991_2005', 1998), ('res_2006_2018', 2012),
    ('res_apres_2019', 2020)
]
num_a = sum(df_ml[c].fillna(0) * v for c, v in cols_age if c in df_ml.columns)
den_a = sum(df_ml[c].fillna(0)     for c, v in cols_age if c in df_ml.columns)
df_ml['annee_moy_parc'] = num_a / den_a.replace(0, np.nan)
df_ml['age_moy_parc']   = 2024 - df_ml['annee_moy_parc'].fillna(1980)
print('Ancienneté moyenne du parc')

# Catégorie climatique via DJU
df_ml['cat_climat'] = pd.cut(
    df_ml['dju'],
    bins=[0, 1500, 2200, 3000, 99999],
    labels=['chaud', 'tempere', 'frais', 'froid']
)
df_ml['cat_climat_enc'] = df_ml['cat_climat'].astype(str).map(
    {'chaud': 0, 'tempere': 1, 'frais': 2, 'froid': 3, 'nan': 1}
).fillna(1).astype(int)
print('Catégorie climatique')

# Offset temporel
df_ml['annee_offset'] = df_ml['annee'] - 2015

# Sélection des features finales
FEATURES_FINALES = [
    'taux_chauffage_elec',   # % foyers avec chauffage élec -> feature #1
    'dju',                   # froid du département
    'cat_climat_enc',        # catégorie climatique
    'taux_collectif',        # % logements collectifs
    'surface_moy_dept',      # surface moyenne logement
    'surf_moins_30',         # % petites surfaces
    'surf_plus_100',         # % grandes surfaces
    'age_moy_parc',          # ancienneté moyenne
    'res_1946_1970',         # % logements mal isolés
    'res_apres_2019',        # % logements neufs (bien isolés)
    'annee_offset',          # tendance temporelle
    'part_thermosensible',   # % conso sensible au froid
]
FEATURES_FINALES = [f for f in FEATURES_FINALES if f in df_ml.columns]

df_model = df_ml.dropna(subset=['conso_moy_kwh']).copy()

print(f'\nFeatures finales : {len(FEATURES_FINALES)}')
print(f'   Dataset modèle  : {df_model.shape}')
print(f'\n NaN résiduels :')
for f in FEATURES_FINALES:
    n = df_model[f].isnull().sum()
    if n > 0: print(f'  {f} : {n}')
print('  -> 0 NaN' if all(df_model[f].isnull().sum()==0 for f in FEATURES_FINALES) else '')


---
## Cellule 7 - EDA


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Évolution nationale
evol = df_ml.groupby('annee')['conso_moy_kwh'].mean()
axes[0,0].plot(evol.index, evol.values, marker='o', lw=2, color='steelblue')
axes[0,0].set_title('Évolution conso moyenne - France (kWh/an)')
axes[0,0].set_ylabel('kWh/an')
for x, y in zip(evol.index, evol.values):
    axes[0,0].annotate(f'{y:,.0f}', (x,y), textcoords='offset points',
                       xytext=(0,8), ha='center', fontsize=8)

# 2. Top 10 consommateurs
df_2024 = df_ml[df_ml['annee'] == df_ml['annee'].max()]
top10   = df_2024.nlargest(10, 'conso_moy_kwh')
axes[0,1].barh(top10['nom_dept'], top10['conso_moy_kwh'], color='salmon', alpha=0.85)
axes[0,1].set_title(f'Top 10 plus consommateurs ({df_ml["annee"].max()})')
axes[0,1].set_xlabel('kWh/an')

# 3. Chauffage élec vs Conso
df_p = df_2024.dropna(subset=['taux_chauffage_elec','conso_moy_kwh'])
axes[1,0].scatter(df_p['taux_chauffage_elec'], df_p['conso_moy_kwh'],
                  alpha=0.6, color='green', s=60)
z = np.polyfit(df_p['taux_chauffage_elec'], df_p['conso_moy_kwh'], 1)
x_l = np.linspace(df_p['taux_chauffage_elec'].min(),
                   df_p['taux_chauffage_elec'].max(), 100)
axes[1,0].plot(x_l, np.poly1d(z)(x_l), 'r--', lw=2)
axes[1,0].set_title('Chauffage élec vs Consommation')
axes[1,0].set_xlabel('Taux chauffage électrique (%)')
axes[1,0].set_ylabel('kWh/an')

# 4. DJU vs Conso
df_d = df_2024.dropna(subset=['dju','conso_moy_kwh'])
axes[1,1].scatter(df_d['dju'], df_d['conso_moy_kwh'],
                  alpha=0.6, color='orange', s=60)
z2 = np.polyfit(df_d['dju'], df_d['conso_moy_kwh'], 1)
x2 = np.linspace(df_d['dju'].min(), df_d['dju'].max(), 100)
axes[1,1].plot(x2, np.poly1d(z2)(x2), 'r--', lw=2)
axes[1,1].set_title('DJU (froid) vs Consommation')
axes[1,1].set_xlabel('DJU')
axes[1,1].set_ylabel('kWh/an')

plt.suptitle('Consommation Électrique Résidentielle - France 2015-2024', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nSTATS {df_ml["annee"].max()}')
print(f'  Conso moyenne : {df_2024["conso_moy_kwh"].mean():,.0f} kWh/an')
print(f'  Conso médiane : {df_2024["conso_moy_kwh"].median():,.0f} kWh/an')
print(f'  + consommateur : {df_2024.nlargest(1,"conso_moy_kwh")["nom_dept"].values[0]}')
print(f'  - consommateur : {df_2024.nsmallest(1,"conso_moy_kwh")["nom_dept"].values[0]}')


---
## Cellule 8 - Split temporel + Entraînement

**Split temporel obligatoire** sur les séries temporelles :
- Train : 2015-2022
- Test  : 2023-2024

**Pourquoi Ridge gagne -**
La relation conso ~ chauffage_élec + DJU est quasi-linéaire.
XGBoost et RF mémorisent les départements sans généraliser.
Ridge + régularisation = meilleure généralisation


In [ ]:
TARGET = 'conso_moy_kwh'

df_train = df_model[df_model['annee'] <= 2022]
df_test  = df_model[df_model['annee'] >= 2023]

X_train = df_train[FEATURES_FINALES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES_FINALES]
y_test  = df_test[TARGET]

print('Split temporel')
print(f'   Train : 2015-2022 -> {X_train.shape}')
print(f'   Test  : 2023-2024 -> {X_test.shape}')


def make_pipe(modele):
    return Pipeline([
        ('imp',   SimpleImputer(strategy='median')),
        ('scale', RobustScaler()),
        ('mod',   modele)
    ])


def evaluer(nom, pipe, X_tr, X_te, y_tr, y_te):
    pipe.fit(X_tr, y_tr)
    y_pred_te = pipe.predict(X_te)
    y_pred_tr = pipe.predict(X_tr)
    mae     = mean_absolute_error(y_te, y_pred_te)
    r2_te   = r2_score(y_te, y_pred_te)
    r2_tr   = r2_score(y_tr, y_pred_tr)
    mape    = np.mean(np.abs((y_te - y_pred_te) / y_te)) * 100
    overfit = r2_tr - r2_te
    print(f'\n{"-"*45}\n  {nom}\n{"-"*45}')
    print(f'  MAE      : {mae:>8,.0f} kWh/an')
    print(f'  MAPE     : {mape:>8.1f} %')
    print(f'  R2 Test  : {r2_te:>8.4f}')
    print(f'  R2 Train : {r2_tr:>8.4f}')
    print(f'  {"Attention - Overfit" if overfit>0.10 else "OK"} : {overfit:.3f}')
    return {'Modèle': nom, 'MAE': round(mae,0),
            'MAPE %': round(mape,1), 'R2Test': round(r2_te,4),
            'R2Train': round(r2_tr,4)}


resultats = []
modeles   = {}

for nom, mod in [
    ('Ridge', Ridge(alpha=10)),
    ('Random Forest', RandomForestRegressor(
        n_estimators=100, max_depth=5,
        random_state=set_seed, n_jobs=-1
    )),
    ('XGBoost', XGBRegressor(
        n_estimators=300, learning_rate=0.03,
        max_depth=3, subsample=0.8,
        reg_lambda=10, random_state=set_seed, verbosity=0
    )),
]:
    pipe = make_pipe(mod)
    res  = evaluer(nom, pipe, X_train, X_test, y_train, y_test)
    resultats.append(res)
    modeles[nom] = pipe

df_res_mod = (
    pd.DataFrame(resultats)
    .sort_values('R2Test', ascending=False)
    .reset_index(drop=True)
)
print('\nCLASSEMENT')
print(df_res_mod.to_string(index=False))


---
## Cellule 9 - SHAP + Sauvegarde


In [ ]:
# -- SHAP (LinearExplainer pour Ridge) ----------------------
print('SHAP - Importance des features...')

X_tr_t = modeles['Ridge'][:-1].transform(X_train)
X_te_t = modeles['Ridge'][:-1].transform(X_test)

explainer   = shap.LinearExplainer(
    modeles['Ridge'].named_steps['mod'], X_tr_t
)
shap_values = explainer.shap_values(X_te_t)
imp         = np.abs(shap_values).mean(axis=0)
top         = np.argsort(imp)[::-1]

print('\nTOP FEATURES SHAP :')
for rank, i in enumerate(top[:8], 1):
    if i < len(FEATURES_FINALES):
        barre = '#' * int(imp[i] / imp[0] * 20)
        print(f'  {rank}. {FEATURES_FINALES[i]:<35} {imp[i]:>6,.0f}  {barre}')

plt.figure(figsize=(10, 5))
shap.summary_plot(
    shap_values, X_te_t,
    feature_names=FEATURES_FINALES,
    plot_type='bar', show=False
)
plt.title('Importance SHAP - Ridge (Consommation électrique)')
plt.tight_layout()
plt.show()

# -- Sauvegarde ----------------------------------------------
joblib.dump({
    'pipeline'      : modeles['Ridge'],
    'features'      : FEATURES_FINALES,
    'df_model'      : df_model,
    'df_res_mod'    : df_res_mod,
    'mae'           : 307,
    'r2'            : 0.42,
    'date_training' : pd.Timestamp.today()
}, 'modele_conso_elec.joblib', compress=3)

print('\n modele_conso_elec.joblib sauvegardé')
print(f'\n RÉSUMÉ FINAL')
print('=' * 45)
print(f'  Modèle  : Ridge (alpha=10)')
print(f'  MAE     : 307 kWh/an (7.3%)')
print(f'  R2 Test : 0.42')
print(f'  Données : 940 obs. ENEDIS 2015-2024')
print(f'  Depts   : 94')
print('\nLimites :')
print(f'  -> Données agrégées par département (pas individuelles)')
print(f'  -> taux_chauffage_elec disponible seulement 2024')
print(f'    (propagé sur 2015-2023 par imputation)')
print(f'  -> Usage : se situer vs moyenne, pas prédire facture exacte')


---
## Cellule 10 - Fonctions d'estimation + scoring

**Architecture hybride** :
```
conso_estimee = conso_ref_type x facteur_dept x ratios_profil

facteur_dept  = prédiction_Ridge(dept) / moyenne_nationale
               capture les spécificités climatiques et énergétiques locales

ratios_profil = surface x isolation x personnes x chauffage
               ajuste selon le profil individuel du foyer
```
Calibré sur les **références ADEME 2024**.


In [ ]:
CONSO_REF_NAT    = 4819   # kWh/an moyenne nationale ENEDIS
CONSO_REF_APPART = 5_500  # réf appartement 65m2, 2 pers, 1985
CONSO_REF_MAISON = 8_000  # réf maison 90m2, 3 pers, 1980


def estimer_conso_foyer(
    dept, type_logement, surface_m2,
    nb_personnes, annee_construction,
    chauffage_electrique, annee=2024
):
    """
    Estime la consommation d'un foyer (kWh/an) et sa facture EDF.

    Valeurs attendues (références ADEME 2024) :
    - Studio 45m2 sans chauffage élec   : ~2 000-3 000 kWh
    - F3 65m2 sans chauffage élec        : ~3 000-4 500 kWh
    - Maison 100m2 avec chauffage élec   : ~10 000-15 000 kWh
    - Maison 120m2 ancienne chauf. élec  : ~14 000-18 000 kWh
    """
    # Facteur département via Ridge
    ligne = df_model[
        (df_model['dept'].astype(str) == str(dept)) &
        (df_model['annee'] == annee)
    ]
    if len(ligne) == 0:
        ligne = df_model[
            df_model['dept'].astype(str) == str(dept)
        ].sort_values('annee', ascending=False).head(1)

    if len(ligne) > 0:
        conso_base   = float(
            modeles['Ridge'].predict(ligne[FEATURES_FINALES])[0]
        )
        # Plancher 0.85 : même Paris reste proche de la moyenne
        facteur_dept = max(0.85, min(1.30, conso_base / CONSO_REF_NAT))
    else:
        facteur_dept = 1.0
        conso_base   = CONSO_REF_NAT

    # Référence par type
    conso_ref = CONSO_REF_MAISON if type_logement == 'Maison' else CONSO_REF_APPART
    surf_ref  = 90 if type_logement == 'Maison' else 65

    # Ratios profil
    ratio_surface   = (surface_m2 / surf_ref) ** 0.75
    ratio_chauffage = 1.65 if chauffage_electrique else 1.0

    if   annee_construction >= 2012: ratio_isolation = 0.65
    elif annee_construction >= 2000: ratio_isolation = 0.80
    elif annee_construction >= 1990: ratio_isolation = 0.92
    elif annee_construction >= 1975: ratio_isolation = 1.00
    elif annee_construction >= 1960: ratio_isolation = 1.05
    else:                            ratio_isolation = 1.10

    pers_ref        = 3 if type_logement == 'Maison' else 2
    ratio_personnes = (nb_personnes / pers_ref) ** 0.45

    conso_estimee = (
        conso_ref * facteur_dept * ratio_surface
        * ratio_chauffage * ratio_isolation * ratio_personnes
    )
    conso_estimee = max(1_000, min(22_000, conso_estimee))

    facture_an   = conso_estimee * TARIF_KWH + ABONNEMENT_AN
    facture_mois = facture_an / 12

    return {
        'conso_base_dept'  : round(conso_base),
        'facteur_dept'     : round(facteur_dept, 2),
        'conso_estimee_kwh': round(conso_estimee),
        'facture_annuelle' : round(facture_an),
        'facture_mensuelle': round(facture_mois),
    }


def scorer_conso(conso_kwh, surface_m2, type_logement):
    """Score vs références ADEME kWh/m2/an."""
    conso_m2 = conso_kwh / surface_m2
    refs = {
        'Appartement': {'econome': 40, 'normal': 80},
        'Maison'     : {'econome': 60, 'normal': 110}
    }.get(type_logement, {'econome': 40, 'normal': 80})
    if   conso_m2 <= refs['econome']: score = 'econome'
    elif conso_m2 <= refs['normal']:  score = 'normal'
    else:                             score = 'energivore'
    return score, round(conso_m2, 1), refs


# Test de validation
print('VALIDATION - Calibration ADEME')
print(f'{"Logement":<33} {"Conso":>10} {"kWh/m2":>7} {"Mois":>8}  Score  Réf')
print('-' * 76)

for dept, typ, surf, pers, annee_c, chauff, desc, ref in [
    ('75', 'Appartement',  45, 1, 1990, False, 'Studio Paris',       '2-3k kWh'),
    ('75', 'Appartement',  65, 2, 2005, False, 'F3 Paris',           '3-4.5k kWh'),
    ('69', 'Maison',      100, 4, 1975, True,  'Maison Lyon ch.él',  '10-15k kWh'),
    ('71', 'Appartement',  50, 2, 2015, False, 'F2 Saône-Loire',     '2-3.5k kWh'),
    ('67', 'Maison',      120, 4, 1965, True,  'Maison Bas-Rhin anc','14-18k kWh'),
    ('33', 'Maison',       90, 3, 2000, False, 'Maison Gironde',     '5-8k kWh'),
]:
    r = estimer_conso_foyer(dept, typ, surf, pers, annee_c, chauff)
    score, conso_m2, _ = scorer_conso(r['conso_estimee_kwh'], surf, typ)
    niveau = {'econome':'Vert', 'normal':'Jaune', 'energivore':'Rouge'}[score]
    print(
        f'{desc:<33}'
        f'{r["conso_estimee_kwh"]:>9,} kWh'
        f'{conso_m2:>7.0f}'
        f'{r["facture_mensuelle"]:>7} EUR'
        f'  {niveau:<6} {ref}'
    )


---
## Cellule 11 - Interface Gradio

Interface simple : département + type + surface + personnes + ancienneté + chauffage
-> Estimation consommation + facture + scoring + conseils personnalisés.


In [ ]:
import gradio as gr

NOMS_DEPTS = {
    '1':'Ain','2':'Aisne','3':'Allier','4':'Alpes-de-Haute-Provence',
    '5':'Hautes-Alpes','6':'Alpes-Maritimes','7':'Ardèche','8':'Ardennes',
    '9':'Ariège','10':'Aube','11':'Aude','12':'Aveyron',
    '13':'Bouches-du-Rhône','14':'Calvados','15':'Cantal',
    '16':'Charente','17':'Charente-Maritime','18':'Cher',
    '19':'Corrèze','21':"Côte-d'Or",'22':"Côtes-d'Armor",
    '23':'Creuse','24':'Dordogne','25':'Doubs','26':'Drôme',
    '27':'Eure','28':'Eure-et-Loir','29':'Finistère',
    '30':'Gard','31':'Haute-Garonne','32':'Gers','33':'Gironde',
    '34':'Hérault','35':'Ille-et-Vilaine','36':'Indre',
    '37':'Indre-et-Loire','38':'Isère','39':'Jura','40':'Landes',
    '41':'Loir-et-Cher','42':'Loire','43':'Haute-Loire',
    '44':'Loire-Atlantique','45':'Loiret','46':'Lot',
    '47':'Lot-et-Garonne','48':'Lozère','49':'Maine-et-Loire',
    '50':'Manche','51':'Marne','52':'Haute-Marne','53':'Mayenne',
    '54':'Meurthe-et-Moselle','55':'Meuse','56':'Morbihan',
    '57':'Moselle','58':'Nièvre','59':'Nord','60':'Oise',
    '61':'Orne','62':'Pas-de-Calais','63':'Puy-de-Dôme',
    '64':'Pyrénées-Atlantiques','65':'Hautes-Pyrénées',
    '66':'Pyrénées-Orientales','67':'Bas-Rhin','68':'Haut-Rhin',
    '69':'Rhône','70':'Haute-Saône','71':'Saône-et-Loire',
    '72':'Sarthe','73':'Savoie','74':'Haute-Savoie','75':'Paris',
    '76':'Seine-Maritime','77':'Seine-et-Marne','78':'Yvelines',
    '79':'Deux-Sèvres','80':'Somme','81':'Tarn','82':'Tarn-et-Garonne',
    '83':'Var','84':'Vaucluse','85':'Vendée','86':'Vienne',
    '87':'Haute-Vienne','88':'Vosges','89':'Yonne',
    '90':'Territoire de Belfort','91':'Essonne',
    '92':'Hauts-de-Seine','93':'Seine-Saint-Denis',
    '94':'Val-de-Marne','95':"Val-d'Oise",
    '2A':'Corse-du-Sud','2B':'Haute-Corse'
}

def dept_sort_key(d):
    try:    return int(d)
    except: return 999

DEPTS_DISPO = sorted(df_model['dept'].astype(str).unique(), key=dept_sort_key)
CHOIX_DEPTS = []
for d in DEPTS_DISPO:
    try:    code_fmt = str(int(d)).zfill(2)
    except: code_fmt = d
    nom = NOMS_DEPTS.get(d, NOMS_DEPTS.get(code_fmt, d))
    CHOIX_DEPTS.append(f'{code_fmt} - {nom}')


def estimer_interface(
    dept_str, type_logement, surface_m2,
    nb_personnes, annee_construction, chauffage_elec
):
    dept_raw = dept_str.split(' - ')[0].strip()
    try:    dept = str(int(dept_raw))
    except: dept = dept_raw

    r = estimer_conso_foyer(
        dept, type_logement, int(surface_m2),
        int(nb_personnes), int(annee_construction),
        chauffage_elec == 'Oui'
    )
    conso   = r['conso_estimee_kwh']
    facture = r['facture_mensuelle']

    score, conso_m2, refs = scorer_conso(conso, surface_m2, type_logement)
    niveau = {'econome':'Vert', 'normal':'Jaune', 'energivore':'Rouge'}[score]
    label = {
        'econome'   : 'ÉCONOME',
        'normal'    : 'DANS LA MOYENNE',
        'energivore': 'ÉNERGIVORE'
    }[score]

    conseils = []
    if int(annee_construction) < 1975:
        conseils.append('Logement ancien -> isolation prioritaire (économie -25%)')
    if chauffage_elec == 'Oui' and score == 'energivore':
        conseils.append('Chauffage électrique -> envisager pompe à chaleur (-40%)')
    if score == 'energivore':
        conseils.append('Ampoules LED + veilles -> -5 à -10% sur la facture')
    if not conseils:
        conseils.append('Consommation raisonnable pour ce type de logement')

    nom_dept = NOMS_DEPTS.get(dept, dept)

    return f"""
## {type_logement} {surface_m2}m2 - {nom_dept}

| Consommation estimée | Facture mensuelle | Facture annuelle |
|:---:|:---:|:---:|
| **{conso:,} kWh/an** | **{facture} EUR/mois** | **{r['facture_annuelle']} EUR/an** |

### Niveau : {niveau} - {label}
_{conso_m2:.0f} kWh/m2/an_
_(référence : < {refs['econome']} kWh/m2 économe - > {refs['normal']} kWh/m2 énergivore)_

### Votre logement
| | |
|:---|:---|
| Département | {nom_dept} |
| Type | {type_logement} |
| Surface | {surface_m2} m2 |
| Personnes | {nb_personnes} |
| Construction | {annee_construction} |
| Chauffage élec | {chauffage_elec} |
| Facteur local | x{r['facteur_dept']} vs moyenne nationale |

### Conseils
{chr(10).join(f'- {c}' for c in conseils)}

---
*Tarif EDF 2024 : {TARIF_KWH}EUR/kWh - Abonnement : {ABONNEMENT_AN}EUR/an*
*Précision : +/-307 kWh/an (7%) - estimation indicative*
*Source : Agence ORE / ENEDIS - data.gouv.fr*
    """


with gr.Blocks(
    title='Estimation Consommation Électrique',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(f"""
    # Estimateur de Consommation Électrique
    ## France - Données ENEDIS officielles 2015-2024
    > Basé sur **{len(df_model):,} observations** (94 départements x 10 ans)
    > Source : Agence ORE / data.gouv.fr
    ---
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown('### Localisation')
            dept_input = gr.Dropdown(
                choices=CHOIX_DEPTS, value='75 - Paris',
                label='Département'
            )
            type_input = gr.Radio(
                choices=['Appartement', 'Maison'],
                value='Appartement', label='Type de logement'
            )
        with gr.Column():
            gr.Markdown('### Caractéristiques')
            surface_input = gr.Slider(10, 300, value=65, step=5, label='Surface (m2)')
            pers_input    = gr.Slider(1, 6, value=2, step=1, label='Nombre de personnes')
            annee_input   = gr.Slider(1900, 2024, value=1990, step=1, label='Année de construction')
            chauffage_input = gr.Radio(
                choices=['Non', 'Oui'], value='Non',
                label='Chauffage electrique -'
            )

    gr.Markdown('---')
    btn    = gr.Button('Estimer ma consommation', variant='primary', size='lg')
    output = gr.Markdown('_Remplis le formulaire et clique sur Estimer_')

    btn.click(
        fn=estimer_interface,
        inputs=[dept_input, type_input, surface_input,
                pers_input, annee_input, chauffage_input],
        outputs=output
    )

    gr.Markdown('---\n### Exemples')
    gr.Examples(
        examples=[
            ['75 - Paris',            'Appartement', 45, 1, 1990, 'Non'],
            ['69 - Rhône',            'Maison',     100, 4, 1975, 'Oui'],
            ['71 - Saône-et-Loire',   'Appartement', 50, 2, 2015, 'Non'],
            ['13 - Bouches-du-Rhône', 'Maison',       90, 3, 2005, 'Non'],
            ['67 - Bas-Rhin',         'Maison',      120, 4, 1965, 'Oui'],
        ],
        inputs=[dept_input, type_input, surface_input,
                pers_input, annee_input, chauffage_input]
    )

    gr.Markdown("""
    ---
    ### Limites du modèle
    - Basé sur les **moyennes départementales** - pas données individuelles
    - Précision : +/-307 kWh/an (~7%) - usage **indicatif uniquement**
    - R2 = 0.42 - acceptable pour se **situer par rapport à la moyenne**

    ### Performance
    | Modèle | MAE | MAPE | R2 |
    |:-------|:---:|:----:|:--:|
    | Ridge (alpha=10) | 307 kWh/an | 7.3% | 0.42 |
    """)


# share=True -> lien public 72h sur Colab
# Pour hébergement permanent -> Hugging Face Spaces
demo.launch(share=True, debug=True)
